# Digital Twin rApp — Sionna RT + SYS Walkthrough

This notebook **re-runs the exact pipeline from the `dtrapp` codebase** (the
**RT + Sionna SYS** build), stage by stage, on a scene you generated with the
codebase, and **visualizes everything** from the 3D world to per-UE / per-cell
**downlink throughput**.

Every number is produced by calling the codebase's own functions, so the results
match a CLI run:

```
python3 -m dtrapp.runner.cli configs/example.yaml
```

**Pipeline:**

| Stage | What we do | Source of truth |
|------|------------|-----------------|
| 1. Geometry | load the codebase's `scene.xml`, reconstruct the scene extent | `dtrapp.geometry` (offline reuse) |
| 2. Network | regenerate the *same* cells + UEs | `dtrapp.network.RandomNetworkSource` |
| 3. Propagation | ray-trace path gain | `dtrapp.propagation.SionnaPropagationEngine` (Sionna RT) |
| 4. SINR | multi-cell SINR | `dtrapp.kpi.sinr` |
| 5. Throughput | **Sionna SYS link adaptation** (5G-NR MCS / BLER) | `dtrapp.kpi.compute_kpis` → `dtrapp.kpi.throughput` |
| 6. Output | write the CSV/JSON | `dtrapp.runner.output.write_outputs` |

The only difference from the RT + Shannon branch is **stage 5**: throughput comes
from Sionna SYS's `InnerLoopLinkAdaptation` + `PHYAbstraction` instead of the
Shannon bound.

**What you need:** a scene folder produced by the codebase
(`output/scene/scene.xml` + `meshes/`), the **same** config YAML (matching
`seed`), and Sionna installed (`pip install sionna sionna-rt`).

## 0. Setup

We force **CPU** by hiding the GPU (`CUDA_VISIBLE_DEVICES=""`) — this avoids the
common CUDA/GPU initialization errors with Sionna on machines without a usable
GPU. **This must run before Sionna is imported**, so it is the first thing in the
first cell. Then we import everything and point the notebook at your repo + scene
folder. **Edit the three paths.**

In [ ]:
import os

# Force CPU: hide the GPU so Sionna RT / Sionna SYS / torch fall back to CPU.
# MUST be set before importing sionna. Comment out to use the GPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")  # quiet TF logging if used

# Install dependencies if missing (uncomment to run once):
# %pip install sionna sionna-rt numpy pyyaml matplotlib pandas

%matplotlib inline
import sys
import glob
import math

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image

# ----------------------- EDIT THESE THREE PATHS -----------------------
# Repo root = the folder that contains the `dtrapp/` package and `configs/`.
REPO_ROOT = os.path.abspath(os.environ.get("DTRAPP_REPO", ".."))
# The scene folder produced by the codebase (contains scene.xml + meshes/).
SCENE_DIR = os.path.join(REPO_ROOT, "output", "scene")
# The SAME config YAML used to generate that scene (the seed must match!).
CONFIG_PATH = os.path.join(REPO_ROOT, "configs", "example.yaml")
# ----------------------------------------------------------------------

# Make the dtrapp package importable from the repo root.
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Set True if you are NOT in a notebook GUI (renders images instead of opening
# the interactive 3D widget, which needs Jupyter ipywidgets in a browser).
no_preview = False

print("REPO_ROOT :", REPO_ROOT)
print("SCENE_DIR :", SCENE_DIR)
print("CONFIG    :", CONFIG_PATH)
print("CUDA_VISIBLE_DEVICES =", repr(os.environ.get("CUDA_VISIBLE_DEVICES")))
assert os.path.exists(os.path.join(SCENE_DIR, "scene.xml")), (
    "scene.xml not found - generate it first, e.g. "
    "`python3 -m dtrapp.runner.cli configs/example.yaml`."
)

## 1. Load the scenario config

The same `SimulationConfig` the codebase uses. Note the two SYS-specific knobs:
`bler_target` (link-adaptation BLER target) and `mcs_table_index`
(`1` = up to 64QAM, `2` = up to 256QAM).

In [ ]:
from dtrapp.config import SimulationConfig

config = SimulationConfig.from_yaml(CONFIG_PATH)

print("Loaded config:")
for k, v in config.to_dict().items():
    print(f"  {k}: {v}")

## 2. Stage 1 — load & visualize the 3D scene

We load the codebase's `scene.xml` into Sionna RT and reconstruct the scene
extent (min/max x,y over all building footprints) directly from the mesh files —
exactly how `dtrapp.geometry.scene_builder` computes the rectangle the network
generator uses, so the regenerated network below is identical.

In [ ]:
import sionna
from sionna.rt import load_scene, Camera

scene = load_scene(os.path.join(SCENE_DIR, "scene.xml"))
print("Scene loaded. Objects (merged by material):", len(scene.objects))


def read_ply(path):
    """Minimal ASCII-PLY reader for the meshes written by dtrapp."""
    with open(path, "r") as fh:
        lines = fh.read().splitlines()
    n_vert, header_end = 0, 0
    for i, ln in enumerate(lines):
        if ln.startswith("element vertex"):
            n_vert = int(ln.split()[-1])
        if ln.strip() == "end_header":
            header_end = i + 1
            break
    verts = [tuple(float(v) for v in ln.split()[:3])
             for ln in lines[header_end:header_end + n_vert]]
    return np.array(verts, dtype=float)


# Reconstruct the scene extent EXACTLY as the codebase does: min/max x,y over all
# building footprints (the ground plane is excluded).
bldg_files = sorted(glob.glob(os.path.join(SCENE_DIR, "meshes", "bldg-*.ply")))
assert bldg_files, "No building meshes (bldg-*.ply) found in the scene folder."
all_xy = np.vstack([read_ply(p)[:, :2] for p in bldg_files])
extent_m = (
    float(all_xy[:, 0].min()), float(all_xy[:, 1].min()),
    float(all_xy[:, 0].max()), float(all_xy[:, 1].max()),
)
print(f"{len(bldg_files)} buildings")
print("extent_m (min_x, min_y, max_x, max_y) =",
      tuple(round(v, 2) for v in extent_m))

In [ ]:
# A handy aerial camera centered on the scene, reused for all renders.
cx = 0.5 * (extent_m[0] + extent_m[2])
cy = 0.5 * (extent_m[1] + extent_m[3])
span = max(extent_m[2] - extent_m[0], extent_m[3] - extent_m[1])
aerial_cam = Camera(position=[cx, cy - span, span], look_at=[cx, cy, 0.0])

# Interactive 3D viewer (in Jupyter) or a rendered still image (otherwise).
if no_preview:
    scene.render_to_file(camera=aerial_cam, filename="scene_overview.png",
                         resolution=[900, 600])
    display(Image("scene_overview.png"))
else:
    scene.preview()

## 3. Stage 2 — network data (cells + UEs)

Regenerated with the codebase's `RandomNetworkSource`, using the same `config`
and the reconstructed `extent_m`, so it reproduces the identical base stations
and UEs a CLI run would create.

In [ ]:
from dtrapp.network import RandomNetworkSource

network = RandomNetworkSource(config, extent_m).generate()
print(f"{len(network.cells)} cells, {len(network.ues)} UEs\n")

print("First 3 cells:")
for c in network.cells[:3]:
    pos = tuple(round(p, 1) for p in c.position)
    print(f"  {c.cell_id}: pos={pos} az={c.azimuth_deg:.1f}deg "
          f"P={c.tx_power_dbm}dBm f={c.carrier_freq_hz/1e9:.2f}GHz "
          f"BW={c.bandwidth_hz/1e6:.0f}MHz")

print("\nFirst 3 UEs:")
for u in network.ues[:3]:
    pos = tuple(round(p, 1) for p in u.position)
    print(f"  {u.ue_id}: pos={pos} demand={u.traffic_demand_mbps:.1f}Mbps "
          f"NF={u.noise_figure_db}dB")

In [ ]:
# Top-down map: building footprints + base stations (with sector azimuths) + UEs.
def plot_footprints(ax):
    for p in bldg_files:
        v = read_ply(p)
        ring = v[: len(v) // 2, :2]            # bottom ring (z = 0)
        ring = np.vstack([ring, ring[0]])      # close the polygon
        ax.fill(ring[:, 0], ring[:, 1], facecolor="0.85",
                edgecolor="0.5", lw=0.5, zorder=1)


fig, ax = plt.subplots(figsize=(9, 9))
plot_footprints(ax)

ax.scatter([u.position[0] for u in network.ues],
           [u.position[1] for u in network.ues],
           c="tab:blue", s=25, label="UE", zorder=3)

for c in network.cells:
    x, y, _ = c.position
    a = math.radians(c.azimuth_deg)
    ax.scatter([x], [y], c="red", marker="^", s=90, zorder=4)
    ax.arrow(x, y, 25 * math.cos(a), 25 * math.sin(a),
             head_width=6, color="red", zorder=4)
ax.scatter([], [], c="red", marker="^", s=90, label="cell (BS sector)")

ax.set_aspect("equal")
ax.set_xlabel("x (m, East)")
ax.set_ylabel("y (m, North)")
ax.set_title("Top-down: buildings, base stations (sectors), and UEs")
ax.legend(loc="upper right")
plt.show()

## 4. Stage 3 — Sionna RT propagation (path gain)

The codebase's `SionnaPropagationEngine` places one transmitter per cell and one
receiver per UE, ray-traces, and reduces to a per-link **path-gain matrix**
`(num_ues, num_cells)` in dB — the input to the SINR stage.

> On CPU this is the slow step. Keep the scene small.

In [ ]:
from dtrapp.propagation import SionnaPropagationEngine

engine = SionnaPropagationEngine(os.path.join(SCENE_DIR, "scene.xml"), config)
path_gain_db = engine.compute_path_gain(network)   # (num_ues, num_cells), dB

print("path_gain_db shape:", path_gain_db.shape)
print("range: %.1f .. %.1f dB" % (path_gain_db.min(), path_gain_db.max()))

In [ ]:
# Heatmap of the per-link path gain.
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(path_gain_db, aspect="auto", cmap="viridis")
ax.set_xlabel("cell index")
ax.set_ylabel("UE index")
ax.set_xticks(range(len(network.cells)))
ax.set_xticklabels([c.cell_id for c in network.cells], rotation=90, fontsize=7)
ax.set_title("Per-link path gain (dB)")
fig.colorbar(im, label="path gain (dB)")
plt.show()

In [ ]:
# Visualize the actual ray paths. The engine leaves its transmitters/receivers
# in place after computing the gain, so we reuse its prepared scene.
from sionna.rt import PathSolver

viz_scene = engine._scene
paths = PathSolver()(viz_scene, max_depth=config.max_depth)

if no_preview:
    viz_scene.render_to_file(camera=aerial_cam, paths=paths,
                             filename="scene_paths.png", resolution=[900, 600])
    display(Image("scene_paths.png"))
else:
    viz_scene.preview(paths=paths)

# Labeled top-down map: every transmitter (cell) and receiver (UE) annotated
# with its ID, so you can cross-reference them with the throughput table below.
fig, ax = plt.subplots(figsize=(11, 11))
plot_footprints(ax)

for u in network.ues:
    ax.scatter([u.position[0]], [u.position[1]], c="tab:blue", s=30, zorder=3)
    ax.annotate(u.ue_id, (u.position[0], u.position[1]),
                textcoords="offset points", xytext=(3, 3),
                fontsize=7, color="tab:blue", zorder=5)

for c in network.cells:
    x, y, _ = c.position
    a = math.radians(c.azimuth_deg)
    ax.scatter([x], [y], c="red", marker="^", s=110, zorder=4)
    ax.annotate("", xy=(x + 25 * math.cos(a), y + 25 * math.sin(a)), xytext=(x, y),
                arrowprops=dict(arrowstyle="->", color="red"), zorder=4)
    ax.annotate(c.cell_id, (x + 33 * math.cos(a), y + 33 * math.sin(a)),
                fontsize=8, fontweight="bold", color="darkred",
                ha="center", va="center", zorder=6)

ax.set_aspect("equal")
ax.set_xlabel("x (m, East)")
ax.set_ylabel("y (m, North)")
ax.set_title("Transmitter (cell) and receiver (UE) IDs\n"
             "match these IDs to the per-UE throughput table below")
plt.show()

## 5. Stages 4-5 — SINR & throughput (Sionna SYS)

`compute_kpis` computes the multi-cell SINR (signal / (inter-cell interference +
noise)) and then maps it to throughput with **Sionna SYS**: for each UE,
`InnerLoopLinkAdaptation` picks the highest 5G-NR MCS whose BLER stays within
`bler_target`, and the resulting (capped) spectral efficiency × bandwidth-share
gives the rate. This needs Sionna SYS installed.

In [ ]:
from dtrapp.kpi import compute_kpis

result = compute_kpis(network, path_gain_db, config)

try:
    import pandas as pd
    ue_df = pd.DataFrame(result.ue_rows())
    cell_df = pd.DataFrame(result.cell_rows())
    print("Per-UE KPIs (first 10 rows):")
    display(ue_df.head(10))
    print("Per-cell KPIs:")
    display(cell_df)
    print("Mean UE throughput: %.2f Mbps" % ue_df["throughput_mbps"].mean())
except ImportError:
    ue_df = cell_df = None
    for u in result.ues[:10]:
        print(u)

In [ ]:
# Distributions of SINR and throughput across UEs.
sinr = np.array([u.sinr_db for u in result.ues])
tput = np.array([u.throughput_mbps for u in result.ues])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(sinr, bins=15, color="tab:orange", edgecolor="k")
axes[0].set_xlabel("SINR (dB)"); axes[0].set_ylabel("# UEs")
axes[0].set_title("SINR distribution")
axes[1].hist(tput, bins=15, color="tab:green", edgecolor="k")
axes[1].set_xlabel("throughput (Mbps)"); axes[1].set_ylabel("# UEs")
axes[1].set_title("UE throughput distribution (Sionna SYS)")
plt.tight_layout(); plt.show()

In [ ]:
# Map: each UE colored by its throughput, labeled with id + Mbps, and linked
# to its serving cell (also labeled).
cell_pos = {c.cell_id: c.position for c in network.cells}

fig, ax = plt.subplots(figsize=(11, 11))
plot_footprints(ax)
for u in result.ues:
    cp = cell_pos[u.serving_cell]
    ax.plot([u.x, cp[0]], [u.y, cp[1]], color="0.7", lw=0.5, zorder=2)
sc = ax.scatter([u.x for u in result.ues], [u.y for u in result.ues],
                c=tput, cmap="viridis", s=45, zorder=3)
for u in result.ues:
    ax.annotate(f"{u.ue_id} ({u.throughput_mbps:.1f})", (u.x, u.y),
                textcoords="offset points", xytext=(3, 3),
                fontsize=6, color="0.2", zorder=5)

for c in network.cells:
    x, y, _ = c.position
    a = math.radians(c.azimuth_deg)
    ax.scatter([x], [y], c="red", marker="^", s=90, zorder=4)
    ax.annotate(c.cell_id, (x + 30 * math.cos(a), y + 30 * math.sin(a)),
                fontsize=8, fontweight="bold", color="darkred",
                ha="center", va="center", zorder=6)

ax.set_aspect("equal")
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
ax.set_title("UEs colored by throughput (label = ue_id and Mbps), "
             "linked to serving cell")
fig.colorbar(sc, label="throughput (Mbps)")
plt.show()

In [ ]:
# Per-cell view: attached UEs and aggregate throughput.
ids = [c.cell_id for c in result.cells]
att = [c.num_attached for c in result.cells]
cmb = [c.throughput_mbps for c in result.cells]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(ids, att, color="tab:purple")
axes[0].set_title("UEs attached per cell"); axes[0].tick_params(axis="x", rotation=90)
axes[1].bar(ids, cmb, color="tab:green")
axes[1].set_title("Aggregate throughput per cell (Mbps)")
axes[1].tick_params(axis="x", rotation=90)
plt.tight_layout(); plt.show()

## 5b. Compare two UEs, factor-by-factor

Throughput here factorises as

\[
\text{throughput} = \underbrace{\frac{\text{cell bandwidth}}{\text{UEs sharing the cell}}}_{\text{resource sharing}}
\; \times \;
\underbrace{\text{SE}_{\text{MCS}}\,(1-\text{BLER})}_{\text{Sionna SYS link adaptation}}
\]

The cell below weighs every factor for two UEs (default `ue6` vs `ue7`) and then
splits the throughput ratio into its *resource-sharing* part and its *link-
adaptation* part, so you can see which one dominates. The goodput spectral
efficiency is read back from the SYS result (`throughput / bandwidth_share`), so
it reflects the actual MCS the link adaptation chose. Edit `UE_A` / `UE_B`.

In [ ]:
from dtrapp.kpi.sinr import (compute_received_power_dbm, associate_cells, dbm_to_mw)

UE_A, UE_B = "ue6", "ue7"          # <-- compare any two UEs

cells = network.cells
tx_power = np.array([c.tx_power_dbm for c in cells])
bw = np.array([c.bandwidth_hz for c in cells])
rx = compute_received_power_dbm(path_gain_db, tx_power)
serving = associate_cells(rx)
attached = np.bincount(serving, minlength=len(cells))
ue_idx = {u.ue_id: i for i, u in enumerate(network.ues)}
res_by_id = {u.ue_id: u for u in result.ues}


def metrics(uid):
    i = ue_idx[uid]; s = int(serving[i]); ue = network.ues[i]; sc = cells[s]
    r = res_by_id[uid]
    dist = float(np.linalg.norm(np.array(ue.position) - np.array(sc.position)))
    rx_mw = dbm_to_mw(rx[i]); sig = float(rx_mw[s]); interf = float(rx_mw.sum() - sig)
    interf_dbm = float(10 * np.log10(interf)) if interf > 0 else float("-inf")
    share = float(bw[s] / attached[s] / 1e6)              # MHz
    se_eff = r.throughput_mbps / share if share > 0 else 0.0   # realised goodput SE
    return dict(serving=sc.cell_id, dist=dist, pg=float(path_gain_db[i, s]),
                interf=interf_dbm, sinr=r.sinr_db, se=float(se_eff),
                load=int(attached[s]), share=share, tput=float(r.throughput_mbps))


a, b = metrics(UE_A), metrics(UE_B)

rows = [
    ("serving cell",          a["serving"], b["serving"], None,   ""),
    ("distance to cell (m)",  a["dist"],    b["dist"],    "low",  "less free-space loss"),
    ("path gain (dB)",        a["pg"],      b["pg"],      "high", "antenna beam + LOS + distance"),
    ("interference (dBm)",    a["interf"],  b["interf"],  "low",  "from all other cells"),
    ("SINR (dB)",             a["sinr"],    b["sinr"],    "high", "signal / (interf + noise)"),
    ("goodput SE (b/s/Hz)",   a["se"],      b["se"],      "high", "MCS x coderate x (1-BLER)"),
    ("cell load (UEs)",       a["load"],    b["load"],    "low",  "fewer sharers = bigger slice"),
    ("bandwidth share (MHz)", a["share"],   b["share"],   "high", ""),
    ("THROUGHPUT (Mbps)",     a["tput"],    b["tput"],    "high", ""),
]

print(f"{'factor':<24}{UE_A:>12}{UE_B:>12}   better")
print("-" * 66)
for name, va, vb, pref, note in rows:
    if pref is None:
        win = ""
    elif pref == "high":
        win = UE_A if va > vb else (UE_B if vb > va else "tie")
    else:
        win = UE_A if va < vb else (UE_B if vb < va else "tie")
    fa = va if isinstance(va, str) else f"{va:.2f}"
    fb = vb if isinstance(vb, str) else f"{vb:.2f}"
    print(f"{name:<24}{fa:>12}{fb:>12}   {win:<5} {note}")

share_ratio = a["share"] / max(b["share"], 1e-12)
se_ratio = a["se"] / max(b["se"], 1e-12)
tput_ratio = a["tput"] / max(b["tput"], 1e-12)
dom = ("resource sharing (cell load)"
       if abs(np.log(max(share_ratio, 1e-12))) >= abs(np.log(max(se_ratio, 1e-12)))
       else "link adaptation (SINR -> MCS)")
print(f"\nThroughput ratio  {UE_A} / {UE_B} = {tput_ratio:.2f}x, and it splits as:")
print(f"  - resource sharing : {UE_A} gets {share_ratio:.2f}x the bandwidth slice of {UE_B}")
print(f"  - link adaptation  : {UE_A} gets {se_ratio:.2f}x the spectral efficiency of {UE_B}")
print(f"  => dominant factor : {dom}")

## 6. (Bonus) Coverage radio map

A `RadioMapSolver` sweeps a horizontal plane and computes path gain / SINR
everywhere — an illustrative continuous coverage view (the authoritative per-UE
numbers are the ones from `compute_kpis` above). Slow/memory-heavy on CPU —
lower `samples_per_tx` or raise `cell_size` if needed.

In [ ]:
from sionna.rt import RadioMapSolver

try:
    viz_scene.bandwidth = float(config.bandwidth_hz)
    viz_scene.temperature = float(config.temperature_k)
except Exception as e:
    print("note: could not set bandwidth/temperature on scene:", e)

rm = RadioMapSolver()(
    viz_scene,
    max_depth=config.max_depth,
    cell_size=(5.0, 5.0),
    center=[cx, cy, config.ue_height_m],
    size=[extent_m[2] - extent_m[0] + 40.0, extent_m[3] - extent_m[1] + 40.0],
    orientation=[0.0, 0.0, 0.0],
    samples_per_tx=10 ** 6,
)

rm.show(metric="path_gain"); plt.show()
rm.show(metric="sinr"); plt.show()

In [ ]:
# Overlay the SINR coverage map on the 3D scene.
if no_preview:
    viz_scene.render_to_file(camera=aerial_cam, radio_map=rm, rm_metric="sinr",
                             filename="radio_map_sinr.png", resolution=[900, 600])
    display(Image("radio_map_sinr.png"))
else:
    viz_scene.preview(radio_map=rm, rm_metric="sinr")

## 7. Stage 6 — write the output files

Writes the same artifacts the CLI produces: `ue_throughput.csv`,
`cell_throughput.csv`, and `throughput.json` in the config's `output_dir`.

In [ ]:
from dtrapp.runner.output import write_outputs

written = write_outputs(result, config)
for p in written:
    print("wrote", p)

try:
    import pandas as pd
    display(pd.read_csv(os.path.join(config.output_dir, "ue_throughput.csv")).head())
except Exception as e:
    print("(install pandas to preview the CSV)", e)

## Notes & troubleshooting

- **GPU / CUDA errors.** The first cell sets `os.environ["CUDA_VISIBLE_DEVICES"]
  = ""` *before* importing Sionna, forcing CPU and avoiding CUDA init errors on
  machines without a usable GPU. To use the GPU, comment that line out. It must
  stay above any `import sionna`; if you change it, **Kernel → Restart** first.
- **Throughput model.** Stage 5 uses Sionna SYS (`InnerLoopLinkAdaptation` +
  `PHYAbstraction`). Tune it via the config: `bler_target` and `mcs_table_index`
  (`1` = up to 64QAM, `2` = up to 256QAM). Requires `pip install sionna`.
- **Exactness.** Network, path gain, SINR and throughput all come from the
  codebase's own functions, so results match a CLI run. Only `extent_m` is
  reconstructed from the mesh vertices (exact to sub-micron).
- **"Widget manager" error (remote/SSH).** `scene.preview()` needs Jupyter
  ipywidgets in a browser. If it errors, set `no_preview = True` to render static
  PNGs instead.
- **Speed (CPU).** The path-solve and radio map are the heavy steps. Shrink the
  bbox, reduce `num_ues`, lower `max_depth`, raise `cell_size`, or reduce
  `samples_per_tx`.